# **Remote Tracking (Colab / GPU)**

This notebook runs the full tracking pipeline (pitch + players/ball) on a video stored in Google Drive using the remote-trained YOLO models under `models/`.

It assumes you have already trained and promoted the **best** pitch and player models with `train/train_remote.ipynb`, so that:
- `models/pitch_detection_model_best/best.pt`
- `models/player_detection_model_best/best.pt`

exist under your Drive project folder (`/content/drive/MyDrive/football_tracking_remote`).

In [ ]:
# Environment & paths (Google Colab + Drive)
import os

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/football_tracking_remote"
else:
    # Fallback when running locally (e.g. in VS Code)
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

MODELS_DIR = os.path.join(BASE_DIR, "models")
FOOTAGE_DIR = os.path.join(BASE_DIR, "footage")
TRACK_OUTPUT_DIR = os.path.join(BASE_DIR, "track_output")

os.makedirs(TRACK_OUTPUT_DIR, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("FOOTAGE_DIR:", FOOTAGE_DIR)
print("TRACK_OUTPUT_DIR:", TRACK_OUTPUT_DIR)

In [ ]:
# Install core dependencies in Colab
%pip install -q ultralytics supervision tqdm python-dotenv

In [ ]:
# Imports and repo setup
from ultralytics import YOLO
import supervision as sv
from tqdm import tqdm
import numpy as np
import os
import sys

# Try to locate the cloned repo root to import project modules
# In Colab this is typically something like /content/FootballTrackingDataGeneration-main
GUESS_REPO_ROOT = "/content/FootballTrackingDataGeneration-main"
if os.path.exists(GUESS_REPO_ROOT):
    REPO_ROOT = GUESS_REPO_ROOT
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from sports.common.team import TeamClassifier
from sports.configs.soccer import SoccerPitchConfiguration
from sports.common.view import ViewTransformer

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# Load remote-trained YOLO models (best weights if available)
PITCH_MODEL_PATH = os.path.join(MODELS_DIR, "pitch_detection_model_best", "best.pt")
if not os.path.exists(PITCH_MODEL_PATH):
    PITCH_MODEL_PATH = os.path.join(MODELS_DIR, "pitch_detection_model", "weights", "best.pt")

PLAYER_MODEL_PATH = os.path.join(MODELS_DIR, "player_detection_model_best", "best.pt")
if not os.path.exists(PLAYER_MODEL_PATH):
    PLAYER_MODEL_PATH = os.path.join(MODELS_DIR, "player_detection_model", "weights", "best.pt")

print("Pitch model path:", PITCH_MODEL_PATH)
print("Player model path:", PLAYER_MODEL_PATH)

PITCH_DETECTION_MODEL = YOLO(PITCH_MODEL_PATH)
PLAYER_DETECTION_MODEL = YOLO(PLAYER_MODEL_PATH)

In [ ]:
# Core configuration: teams/video and switches
GENERATE_VIDEO = 1
TRACK_TEAMS = 1

# Video file to track (stored under BASE_DIR/footage)
VIDEO_FILE = "sample_30.mov"
SOURCE_VIDEO_PATH = os.path.join(FOOTAGE_DIR, VIDEO_FILE)

print("Using video:", SOURCE_VIDEO_PATH)

In [ ]:
# Object classes and annotators (same as local tracking notebook)
objects = {
    "ball": 0,
    "goalkeeper": 1,
    "player": 2,
    "referee": 3,
}

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    thickness=2,
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    text_color=sv.Color.from_hex("#000000"),
    text_position=sv.Position.BOTTOM_CENTER,
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex("#FFD700"),
    base=25,
    height=21,
    outline_thickness=1,
)
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(["#FF8C00", "#00BFFF", "#FF1493", "#FFD700"]),
    thickness=2,
)

In [ ]:
# Identify goalkeeper team by proximity to team centroids
import numpy as np  # ensure numpy is available in this cell


def resolve_goalkeepers_team_id(players, goalkeepers):
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)
    goalkeepers_team_id = []
    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(goalkeeper_xy - team_0_centroid)
        dist_1 = np.linalg.norm(goalkeeper_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

In [ ]:
# Compute detections, assign teams, and transform to pitch coordinates

def get_detections(frame, detections, key_points, tracker, team_classifier):
    CONFIG = SoccerPitchConfiguration()

    # Organize detections
    ball_detections = detections[detections.class_id == objects["ball"]]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    all_detections = detections[detections.class_id != objects["ball"]]
    all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
    all_detections = tracker.update_with_detections(detections=all_detections)

    goalkeepers_detections = all_detections[all_detections.class_id == objects["goalkeeper"]]
    players_detections = all_detections[all_detections.class_id == objects["player"]]

    if team_classifier:
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)

    goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
        players_detections, goalkeepers_detections
    )

    # Adjust points to 2D pitch
    filter_mask = key_points.confidence[0] > 0.5
    frame_reference_points = key_points.xy[0][filter_mask]
    pitch_reference_points = np.array(CONFIG.vertices)[filter_mask]

    transformer = ViewTransformer(
        source=frame_reference_points,
        target=pitch_reference_points,
    )

    frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)
    ball_detections.data["pitch_xy"] = pitch_ball_xy

    frame_goalkeepers_xy = goalkeepers_detections.get_anchors_coordinates(
        sv.Position.BOTTOM_CENTER
    )
    pitch_goalkeepers_xy = transformer.transform_points(points=frame_goalkeepers_xy)
    goalkeepers_detections.data["pitch_xy"] = pitch_goalkeepers_xy

    frame_players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_players_xy = transformer.transform_points(points=frame_players_xy)
    players_detections.data["pitch_xy"] = pitch_players_xy

    # Merge detections
    all_detections = sv.Detections.merge([players_detections, goalkeepers_detections])

    return all_detections, ball_detections

In [ ]:
# Learn team colors from a stride of frames

def generate_team_model(video_path, player_model):
    STRIDE = 30

    frame_generator = sv.get_video_frames_generator(
        source_path=video_path, stride=STRIDE
    )

    crops = []
    for frame in tqdm(frame_generator, desc="collecting crops"):
        results = player_model(frame, conf=0.3)
        result = results[0]
        detections = sv.Detections.from_ultralytics(result)
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
        crops += players_crops

    team_classifier = TeamClassifier()
    team_classifier.fit(crops)

    return team_classifier

In [ ]:
# Save tracking results (players + ball) to CSV under Drive

def save_tracking_results(players, ball, frames):
    csv = "Frame,Object,Object ID,Team,X1,Y1,X2,Y2,X_Pitch,Y_Pitch\n"
    for frame in range(1, frames):
        for player_id in players:
            player_data = players[player_id]
            if str(frame) in player_data:
                row = player_data[str(frame)]
                csv += (
                    f"{frame},player,{player_id},{row['Team']},{row['X1']},{row['Y1']},{row['X2']},{row['Y2']},{row['X_Pitch']},{row['Y_Pitch']}\n"
                )
        if str(frame) in ball:
            row = ball[str(frame)]
            csv += (
                f"{frame},ball,,,{row['X1']},{row['Y1']},{row['X2']},{row['Y2']},{row['X_Pitch']},{row['Y_Pitch']}\n"
            )

    os.makedirs(TRACK_OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(TRACK_OUTPUT_DIR, VIDEO_FILE + ".csv")
    with open(output_path, "w") as file:
        file.write(csv)
    print("Saved tracking CSV to", os.path.abspath(output_path))

In [ ]:
# Optional: learn team classifier from the video
team_classifier = None
if TRACK_TEAMS:
    team_classifier = generate_team_model(SOURCE_VIDEO_PATH, PLAYER_DETECTION_MODEL)

In [ ]:
# Main tracking loop: runs on SOURCE_VIDEO_PATH and writes video+CSV to Drive
CONFIG = SoccerPitchConfiguration()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

tracker = sv.ByteTrack()
tracker.reset()

frame_number = 1
video_info = sv.VideoInfo.from_video_path(video_path=SOURCE_VIDEO_PATH)
output_video_path = os.path.join(
    TRACK_OUTPUT_DIR, VIDEO_FILE.rsplit(".", 1)[0] + "_tracked.mp4"
)
players = {}
ball = {}

with sv.VideoSink(target_path=output_video_path, video_info=video_info) as sink:
    for frame in tqdm(frame_generator, desc="Collecting Tracking Data..."):
        # Player detection (Ultralytics YOLO detect)
        player_results = PLAYER_DETECTION_MODEL(frame, conf=0.3)
        player_result = player_results[0]
        detections = sv.Detections.from_ultralytics(player_result)

        # Pitch keypoints (Ultralytics YOLO pose)
        pitch_results = PITCH_DETECTION_MODEL(frame, conf=0.3)
        pitch_result = pitch_results[0]
        key_points = sv.KeyPoints.from_ultralytics(pitch_result)

        # Organize detections
        all_detections, ball_detections = get_detections(
            frame, detections, key_points, tracker, team_classifier
        )
        object_ids = all_detections.tracker_id
        team_ids = all_detections.class_id
        object_types = all_detections.data["class_name"]
        pitch_xys = all_detections.data["pitch_xy"]
        ball_pitch_xys = ball_detections.data["pitch_xy"]
        all_detections.class_id = all_detections.class_id.astype(int)

        labels = [f"#{tracker_id}" for tracker_id in all_detections.tracker_id]

        # Store per-frame data
        for idx, xyxy in enumerate(all_detections.xyxy):
            team_id = 0
            if tracker:
                team_id = team_ids[idx]

            object_id = str(object_ids[idx])
            if object_id not in players:
                players[object_id] = {}

            players[object_id][str(frame_number)] = {
                "Object Type": object_types[idx],
                "Team": team_id,
                "X1": xyxy[0],
                "Y1": xyxy[1],
                "X2": xyxy[2],
                "Y2": xyxy[3],
                "X_Pitch": pitch_xys[idx][0],
                "Y_Pitch": pitch_xys[idx][1],
                "Y_MPLSoccer": float(float(pitch_xys[idx][1]) / float(CONFIG.width)),
                "X_MPLSoccer": float(float(pitch_xys[idx][0]) / float(CONFIG.length)),
            }

        if ball_detections.xyxy.shape[0]:
            ball[str(frame_number)] = {
                "X1": ball_detections.xyxy[0][0],
                "Y1": ball_detections.xyxy[0][1],
                "X2": ball_detections.xyxy[0][2],
                "Y2": ball_detections.xyxy[0][3],
                "X_Pitch": ball_pitch_xys[0][0],
                "Y_Pitch": ball_pitch_xys[0][1],
                "Y_MPLSoccer": float(float(ball_pitch_xys[0][1]) / float(CONFIG.width)),
                "X_MPLSoccer": float(float(ball_pitch_xys[0][0]) / float(CONFIG.length)),
            }
        else:
            ball[str(frame_number)] = {
                "X1": 0,
                "Y1": 0,
                "X2": 0,
                "Y2": 0,
                "X_Pitch": 0,
                "Y_Pitch": 0,
                "Y_MPLSoccer": 0,
                "X_MPLSoccer": 0,
            }

        frame_number += 1

        if GENERATE_VIDEO:
            annotated_frame = frame.copy()
            annotated_frame = ellipse_annotator.annotate(
                scene=annotated_frame,
                detections=all_detections,
            )
            annotated_frame = label_annotator.annotate(
                scene=annotated_frame,
                detections=all_detections,
                labels=labels,
            )
            annotated_frame = triangle_annotator.annotate(
                scene=annotated_frame,
                detections=ball_detections,
            )

            sink.write_frame(frame=annotated_frame)

print("Saved annotated video to", os.path.abspath(output_video_path))

In [ ]:
# Save CSV to Drive
save_tracking_results(players, ball, frame_number)